![Databricks Academy](./Includes/images/common/db-academy.png)

# 2 - Developing a Simple Pipeline

In this demonstration, we will create a simple Lakeflow Spark Declarative Pipeline project using the new **Lakeflow Pipeline Editor** with declarative SQL.


### Learning Objectives

By the end of this lesson, you will be able to:
- Describe the SQL syntax used to create a Lakeflow Spark Declarative Pipeline.
- Navigate the Lakeflow Pipeline Editor to modify pipeline settings and ingest the raw data source file(s).
- Create, execute, and monitor a Spark Declarative Pipeline.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](./Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


## A. Classroom Setup

Run the following cell to configure your working environment for this course.

This cell will also reset your `/Volumes/labuser/sdp_1_bronze/source` volume with the JSON files to the starting point, with one JSON file in each directory.

In [0]:
%run ./Includes/Classroom-Setup-REQUIRED

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.8/832.8 kB 12.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-36c5afa3-e1f0-44d3-83be-ce082e463c75
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.67.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-36c5afa3-e1f0-44d3-83be-ce082e463c75
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,

✅ Vocareum workspace detected.
✅ Using existing Vocareum catalog: 'labuser15140516_1778971530'.



  STEP 1: Verifying catalog exists: labuser15140516_1778971530
  Catalog 'labuser15140516_1778971530' exists.

  STEP 2: Setting up 3 schema(s) in catalog: labuser15140516_1778971530
  [1/3] Checking: `labuser15140516_1778971530`.`sdp_1_bronze`... ALREADY EXISTS
  [2/3] Checking: `labuser15140516_1778971530`.`sdp_2_silver`... ALREADY EXISTS
  [3/3] Checking: `labuser15140516_1778971530`.`sdp_3_gold`... ALREADY EXISTS

  COMPLETE: 0 schema(s) created, 3 already existed.



DataFrame[]


  Searching for 'Includes/data' folder...
  Current directory: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines
  Checking: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/Includes/data... FOUND


  STEP 1: Validating volume folder path...
  Found: /Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers

  STEP 2: Scanning for files...
  Found 1 file(s) to delete.

  STEP 3: Deleting files from: /Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers
  [1/1] Deleting: 00.json... DELETED

  COMPLETE: Deleted 1 file(s) from /Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers


  STEP 1: Validating source workspace folder...
  Source folder found: /Workspace/Users/la

Information,Value
Your Catalog:,labuser15140516_1778971530
Bronze Schema:,sdp_1_bronze
Silver Schema:,sdp_2_silver
Gold Schema:,sdp_3_gold
Source Volume:,/Volumes/labuser15140516_1778971530/sdp_1_bronze/source


Compute,Status,Details
Serverless,✓ Match,Version 5


## B. Developing and Running a Spark Declarative Pipeline with the Lakeflow Pipeline Editor


This course includes a simple, pre-configured Spark Declarative Pipeline to explore and modify.

In this section, we will:

- Explore the Lakeflow Pipeline Editor and the declarative SQL syntax
- Modify pipeline settings
- Run the Spark Declarative Pipeline and explore the streaming tables and materialized view.

### B1. Create a Spark Declarative Pipeline From Existing Assets

1. Execute a célula abaixo e **copie o caminho** da célula de saída para o seu volume **labuser_USERNAME.sdp_1_bronze.source**.

   Você precisará deste caminho ao modificar as configurações do seu pipeline.

   Este caminho de volume contém os diretórios **orders**, **status** e **customer**, que possuem os arquivos JSON brutos.

   **EXEMPLO DE CAMINHO**: `/Volumes/labuser/sdp_1_bronze/source`

In [0]:
%python
print(source_volume_path)

/Volumes/labuser15140516_1778971530/sdp_1_bronze/source


*Neste curso, temos arquivos iniciais para você usar em seu pipeline. Esta demonstração utiliza a pasta **2 - Developing a Simple Pipeline Project**.*

2. Para criar o pipeline e adicionar ativos existentes para associá-lo a arquivos de código já disponíveis em seu Workspace (incluindo pastas Git), complete o seguinte:

   a. Para facilitar, abra **Jobs & Pipelines** em uma nova aba:

    - Na barra de navegação principal, clique com o botão direito em **Jobs & Pipelines** e selecione **Abrir em uma nova aba**.

   b. Em **Jobs & Pipelines** selecione **Criar** → **ETL Pipeline**.

   c. Selecione **Configurações (ou o ícone de engrenagem)** e complete o seguinte:

 Seção | Campo | Valor |
---------|-------|---------------|
 **Configurações do pipeline** |  **Nome** | `2 - Developing a Simple Pipeline-adicione-seu-nome` |
 **Local padrão para ativos de dados** | **Catálogo padrão** | Seu catálogo **labuser** |
 **Local padrão para ativos de dados** | **Schema padrão** | Seu schema (banco de dados) **sdp_1_bronze** |

   d. Em **Configurações**, na seção **Ativos de código** selecione **Configurar caminhos** para referenciar nossos arquivos de projeto.

- Para **Pasta raiz do pipeline**: Selecione a pasta **2 - Developing a Simple Pipeline Project** dentro desta pasta do curso e clique em **Selecionar**:
- `.../Build Data Pipelines with Lakeflow Spark Declarative Pipelines/2 - Developing a Simple Pipeline Project`

- **Caminhos do código fonte**: Dentro da mesma pasta raiz acima, selecione a pasta **orders** e clique em **Selecionar**:
- `.../Build Data Pipelines with Lakeflow Spark Declarative Pipelines/2 - Developing a Simple Pipeline Project/orders`

    **NOTA:** Você pode selecionar pastas contendo arquivos SQL e Python para serem executados como parte do pipeline, ou pode fornecer caminhos de arquivos individuais. Os arquivos especificados serão processados quando o pipeline for executado.

**Exemplo**

<img src="./Includes/images/developing-a-simple-pipeline/select_assets.png" alt="Configuração de ativos do pipeline" width="900">

### B2. Explore the Pipeline Editor

1. In the new **Lakeflow Pipeline Editor** tab, select the **orders_pipeline.sql** file.

    In the left navigation pane, confirm you are in the **Pipeline** tab.

    Here you should see your pipeline assets.

<br></br>
![Orders File Directions](./Includes/images/developing-a-simple-pipeline/demo02-sql-orders-files.png)

#### B2.1. Explore the Folder Structure

Your pipeline project contains three folders:

| Folder | Description |
|--------|-------------|
| `exploration` | Sample exploration notebook. **This is excluded from the pipeline** |
| `orders` | Contains the orders pipeline code (`orders_pipeline.sql`) |
| `python_excluded` | Python version of the SQL pipeline code. This course focuses on SQL. Feel free to explore the Python version. |

> **NOTE:** You can structure your pipeline project and files however you would like.

#### B2.2. Incluir ou Excluir Pastas do Pipeline

1. Expanda a pasta `exploration`.
    - Observe o ícone à direita do notebook `sample_exploration` ![Excluir](./Includes/images/developing-a-simple-pipeline/exclude.png)

    - Passe o mouse sobre ele para confirmar que está **excluído** do pipeline.

2. Clique com o botão direito na pasta `exploration`.
    - Observe a opção **Incluir pasta como código fonte do pipeline** (não selecione, não queremos incluir explorações).

    - Você pode alternar para incluir ou excluir a pasta. **Deixe excluída por enquanto.**

3. Neste momento, apenas os arquivos da pasta **orders** estão **incluídos** no pipeline.

#### B2.3. Explorar e Modificar Configurações do Pipeline

1. Selecione o ícone **Configurações** (ícone de engrenagem) na barra de navegação à esquerda, abaixo das abas **Pipeline** e **Todos os Arquivos**.
    - Um painel será aberto à direita.

2. **Configurações do Pipeline** - Revise os detalhes do pipeline:
    - ID do pipeline, tipo, nome, modo, criador, proprietário e executar como.
    - Deixe esses campos como estão.

3. **Ativos de Código** - Esta seção referencia automaticamente todos os arquivos incluídos em seu projeto.
    - **Pasta raiz** - referencia toda a pasta do projeto do pipeline
    - **Código fonte** - lista todos os arquivos incluídos (deve mostrar apenas a pasta `orders`)
    - **NOTA:** Use **Configurar caminhos** para referenciar arquivos que deseja adicionar dentro ou fora da pasta raiz do projeto, se necessário

4. **Local Padrão para Ativos de Dados**

    - Selecione **Editar catálogo e schema** e confirme o seguinte:

 Campo | O que selecionar |
-------|------------------|
 **Catálogo padrão** | Seu catálogo **labuser** |
 **Schema padrão** | **sdp_1_bronze** |


- Se você atualizou esses campos, selecione **Salvar**.

    > **NOTA:** Com Lakeflow Spark Declarative Pipelines, você pode publicar tabelas de streaming e views materializadas em qualquer catálogo e schema. Você não está restrito ao catálogo e schema padrão.

5. **Compute** - Define o compute do pipeline

    a. Selecione o ícone **Editar** e confirme que **Serverless** está selecionado.

    b. Desmarque **Serverless** e revise as opções de compute disponíveis, políticas de cluster e tags.

    c. **Selecione novamente Serverless** e volte para as configurações do pipeline.

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    REQUERIDO - Adicione um Parâmetro de Configuração Apontando para seu Volume de Origem
  </strong>
  <div style="color:#333;">

6. Complete os seguintes passos para adicionar uma variável `source` apontando para seu volume de dados brutos.

   a. Execute a célula abaixo e copie o caminho para seu volume de dados de origem.

   b. Selecione **Adicionar configuração**.

   c. **Chave** = `source`

   d. **Valor** = Cole o caminho do volume do seu `labuser.sdp_1_bronze.source`.
    - Exemplo: `/Volumes/labuser/sdp_1_bronze/source`

   e. Selecione **Salvar**.

**Use parâmetros com pipelines**:
[AWS](https://docs.databricks.com/aws/en/ldp/parameters) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/parameters) |
[GCP](https://docs.databricks.com/gcp/en/ldp/parameters)
  </div>
</div>

In [0]:
%python
print(source_volume_path)

/Volumes/labuser15140516_1778971530/sdp_1_bronze/source



7. **Usage** - Review the **Usage** section. This is where you can assign a budget policy for Serverless compute. Tags are applied to compute activity and logged in your billing records.

8. **Notifications** - Enables you to add notifications on pipeline runs.

9. **Advanced Settings**

     a. Expand **Advanced Settings** and select **Edit advanced settings**. Review the following options:

    - **Channel** - controls the pipeline runtime version. Leave as **Current**.

    - **Event Logs** - optionally write pipeline audit logs, data quality checks, and lineage to a table. Leave this deselected for now.

10. Select anywhere in the code editor to close the **settings** panel.


#### B2.4. Explore the SDP SQL Code
1. Review the SQL code in the **orders_pipeline.sql** file.

> **NOTE:** You can also use Python to build pipelines. That can be found in the **python_excluded** folder. This demonstration focuses on SQL.

<br></br>
##### EXPAND FOR THE ORDERS CODE DETAILS

<details>

###### A. Bronze - Raw Ingestion (Streaming Table)

- Ingests raw JSON files from a Volume using Auto Loader (`STREAM read_files`)
- Adds `processing_time` and `source_file` columns for auditing
- **Incrementally processes only new files** on each pipeline run

###### B. Silver - Cleaned & Typed (Streaming Table)

- Reads incrementally from the bronze streaming table (`FROM STREAM sdp_1_bronze.orders_bronze_demo2`)
- Selects relevant columns and casts `order_timestamp` to `TIMESTAMP`
- Only new rows from bronze flow through

###### C. Gold - Aggregation (Materialized View)

- Aggregates silver into **daily order counts** (`GROUP BY date`)
- Uses a `MATERIALIZED VIEW` instead of a streaming table
- Aggregations require a full table scan. **However, Databricks optimizes recomputation where possible**
- **Incremental refresh for materialized views**:
[AWS](https://docs.databricks.com/aws/en/optimizations/incremental-refresh) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/optimizations/incremental-refresh) |
[GCP](https://docs.databricks.com/gcp/en/optimizations/incremental-refresh)


**Key Differences ST/MV**:
  - Bronze and Silver use `STREAMING TABLE` for incremental processing.
  - Gold uses `MATERIALIZED VIEW` because `GROUP BY` aggregations can't be incrementally computed.

</details>

## C. Run the Spark Declarative Pipeline

### C1. Execução Simulada (Dry Run) do Pipeline
Usando uma execução simulada, você pode **verificar problemas no código-fonte de um pipeline** sem esperar que tabelas sejam criadas ou atualizadas.

Esse recurso é útil ao desenvolver ou testar pipelines, pois permite encontrar e corrigir rapidamente erros no pipeline, como nomes incorretos de tabelas ou colunas.

1. <span style="background-color: #fff3e0; padding: 2px 6px; border-radius: 4px;">Selecione **Execução Simulada** (Dry Run) na barra de navegação superior (se sua tela for pequena, talvez seja necessário selecionar o menu ao lado do botão **Executar pipeline** na barra de navegação superior).</span>

    - Observe que o pipeline muda para o modo **DRY RUN** na janela à esquerda e começa a processar cada etapa (isso leva cerca de ~1 minuto).

    - Após a execução simulada, você deve ver três itens no **Gráfico do Pipeline** (se necessário, selecione o ícone de gráfico na barra de navegação à direita):

      ![Graph](./Includes/images/developing-a-simple-pipeline/dry-run-pipeline-graph.png)

2. Explore a janela inferior e observe as seguintes colunas:

    - **Catálogo** - o catálogo em que cada objeto será escrito

    - **Schema** - o schema em que cada objeto será escrito

    - **Tipo** - o tipo do objeto

      ![Window](./Includes/images/developing-a-simple-pipeline/pipeline-dry-run-window.png)

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Troubleshooting a Pipeline Error
  </strong>
  <div style="color:#333;">

If your pipeline returns an error, verify the following in your <strong>Pipeline Settings</strong>:

  - Default catalog is set to your <strong>labuser</strong> catalog

  - Default schema is set to <strong>sdp_1_bronze</strong>

  - The `source` configuration variable references your volume path: `/Volumes/labuser/sdp_1_bronze/source`

  </div>
</div>

### C2. Execute o Pipeline Spark Declarativo

1. Selecione o menu ao lado do botão **Executar pipeline** na barra de navegação superior.

Você verá duas opções:

 Tipo de Atualização | Materialized View | Streaming Table |
---------------------|------------------|-----------------|
 **Executar pipeline** - Refresh | Atualiza os resultados para refletir os resultados atuais da consulta definida. Irá analisar os custos e realizar uma atualização incremental se for mais eficiente. | Processa novos registros através da lógica definida nas tabelas de streaming e fluxos. |
 **Executar pipeline com atualização completa da tabela** - Atualização completa do pipeline | Atualiza os resultados para refletir os resultados atuais da consulta definida. | Limpa os dados das tabelas de streaming, limpa informações de estado (checkpoints) dos fluxos e reprocessa todos os registros da fonte de dados. |

2. Selecione **Executar pipeline** e monitore cada etapa na janela à direita.


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Information
  </strong>
  <div style="color:#333;">

- A full refresh deletes all objects and checkpoints. Be careful when using.
- To prevent a full refresh from being triggered, set the table property: `pipelines.reset.allowed = false`
- **Pipeline refresh semantics**:
[AWS](https://docs.databricks.com/aws/en/ldp/updates#pipeline-refresh-semantics) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/updates#pipeline-refresh-semantics) |
[GCP](https://docs.databricks.com/gcp/en/ldp/updates#pipeline-refresh-semantics)

  </div>
</div>


### C3. Monitor the Pipeline Run

1. While the pipeline is running (~1 minute), observe the **Pipeline graph** on the right.

    As each object is created, a data flow appears and row counts update as processing completes.

    Expected results:
    - **174 rows** ingested from Raw JSON → Bronze → Silver
    - **7 rows** in the materialized view aggregation

### C4. Explore o Pipeline Concluído

Após a conclusão do pipeline, explore os resultados:

1. Revise o **Gráfico do Pipeline**. O gráfico visualiza todo o fluxo de dados.

2. Revise a janela inferior para detalhes sobre as tabelas e views materializadas criadas, incluindo localizações, durações e contagem de linhas.

   a. <span style="background-color: #fff3e0; padding: 2px 6px; border-radius: 4px;">Selecione a aba **Performance** para visualizar métricas de desempenho, depois retorne para **Tables**.</span>

   b. <span style="background-color: #fff3e0; padding: 2px 6px; border-radius: 4px;">Selecione a tabela de streaming **orders_bronze_demo2** para visualizar seus **dados**, **Métricas da Tabela** e **Performance**.</span>

   c. <span style="background-color: #fff3e0; padding: 2px 6px; border-radius: 4px;">Selecione a seta à esquerda de **All tables** na janela inferior para retornar à lista completa de objetos do pipeline.</span>

   d. <span style="background-color: #fff3e0; padding: 2px 6px; border-radius: 4px;">Selecione a view materializada **gold_orders_by_date_demo2** e confirme que ela resume os dados por data conforme esperado.</span>
      - <span style="background-color: #fff3e0; padding: 2px 6px; border-radius: 4px;">Use a barra de navegação na janela inferior para explorar as abas **Data**, **Columns**, **Métricas da Tabela** e **Performance** da view materializada.</span>

### C5. Explore the DAG (Pipeline Graph)

1. You can also select objects directly within the DAG (Pipeline Graph) to update the details in the bottom window.

### C6. Run the Pipeline Again

1. Select **Run pipeline** again and explore the updated results while it runs.

2. After the second run completes, note that no new rows were added to the streaming tables.

**This is expected since no new files were added to the data source and the checkpoints know the current data was already ingested**.

![Run 2](./Includes/images/developing-a-simple-pipeline/run-2-no-changes.png)

## D. Add a New File to Cloud Storage

1. Run the cell below to add a new JSON file (**01.json**) to your volume at:  `/Volumes/labuser/sdp_1_bronze/source/orders`.

      This will simulate files being added to cloud storage.

In [0]:
%python
## Find data in workspace data folder
data_path = find_folder('Includes/data')

## Land another JSON file to your orders volume
copy_workspace_files_to_volume(
    src_workspace_folder=f'{data_path}/orders',
    target_volume_path=f'{source_volume_path}/orders',
    n=2
)


  Searching for 'Includes/data' folder...
  Current directory: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines
  Checking: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/Includes/data... FOUND


  STEP 1: Validating source workspace folder...
  Source folder found: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/Includes/data/orders

  STEP 2: Checking target volume path...
  Target volume path already exists: /Volumes/labuser15140516_1778971530/sdp_1_bronze/source/orders

  STEP 3: Reading source files...
  Found 5 file(s) in source folder.

  STEP 4

2. Complete the following steps to view the new file in your volume:

   a. Select the **Catalog** icon ![Catalog Icon](./Includes/images/common/catalog_icon.png) from the left navigation pane.

   b. Expand your **labuser.sdp_1_bronze.source** volume.

   c. Expand the **orders** directory.

   d. You should see two files in your volume: **00.json** and **01.json** (refresh if necessary).

3. Run the cell below to view the data in the new **/orders/01.json** file. Notice the following:

   - The **01.json** file contains new orders.
   - The **01.json** file has 25 rows.


In [0]:
%python
spark.sql(f'''
  SELECT *
  FROM json.`{source_volume_path}/orders/01.json`
''').display()

customer_id,notifications,order_id,order_timestamp
23180,Y,75297,1640996822
23082,Y,75298,1641000470
23550,Y,75299,1641000707
23362,Y,75300,1641002550
23210,N,75301,1641003380
23489,N,75302,1641004093
23328,Y,75303,1641004638
23954,Y,75304,1641009857
23648,Y,75305,1641011274
23310,Y,75306,1641015380


4. Go back to the **orders_pipeline.sql** file and select **Run Pipeline** to execute your ETL pipeline again with the new file.

   - Watch the pipeline run and notice only **25 rows** are added to the bronze and silver tables.

   - This happens because:
      - The pipeline has already processed the initial **00.json** file (174 rows)
      - It now reads only the new **01.json** file (25 rows)
      - New rows are appended to the **streaming tables**
      - The materialized view is **incrementally recomputed** using the latest data (should have a total of **8 rows**)

#### Checkpoint
![Final](./Includes/images/developing-a-simple-pipeline/demo2-final.png)

## E. Exploring Your Streaming Tables

### E1. View the Tables

1. View the new streaming tables and materialized view in your catalog. Complete the following:

   a. Select the catalog icon ![Catalog Icon](./Includes/images/common/catalog_icon.png) in the left navigation pane.

   b. Expand your **labuser** catalog.

   c. Expand the schemas **sdp_1_bronze**, **sdp_2_silver**, and **sdp_3_gold**.
      - Notice that the two streaming tables and materialized view are correctly placed in your schemas.

         - Streaming Bronze Table: **labuser.sdp_1_bronze.orders_bronze_demo2**

         - Streaming Silver Table: **labuser.sdp_2_silver.orders_silver_demo2**

         - Gold Materialized View: **labuser.sdp_3_gold.orders_by_date_gold_demo2**

2. Run the cell below to view the data in the **labuser.sdp_1_bronze.orders_bronze_demo2** table.

   Before you run the cell, how many rows should this streaming table have?

   Notice the following:
      - The table contains 199 rows (**00.json** had 174 rows, and **01.json** had 25 rows).
      - In the **source_file** column you can see the exact file the rows were ingested from.
      - In the **processing_time** column you can see the exact time the rows were ingested.

In [0]:
SELECT *
FROM sdp_1_bronze.orders_bronze_demo2;

customer_id,notifications,order_id,order_timestamp,_rescued_data,processing_time,source_file
23094,Y,75123,1640392092,null,2026-05-17T15:00:45.126Z,00.json
23457,N,75124,1640392500,null,2026-05-17T15:00:45.126Z,00.json
23564,Y,75125,1640394862,null,2026-05-17T15:00:45.126Z,00.json
23392,N,75126,1640396067,null,2026-05-17T15:00:45.126Z,00.json
23101,Y,75127,1640399066,null,2026-05-17T15:00:45.126Z,00.json
23466,N,75128,1640404853,null,2026-05-17T15:00:45.126Z,00.json
23834,Y,75129,1640407272,null,2026-05-17T15:00:45.126Z,00.json
23852,Y,75130,1640419989,null,2026-05-17T15:00:45.126Z,00.json
23483,Y,75131,1640422131,null,2026-05-17T15:00:45.126Z,00.json
23821,N,75132,1640423697,null,2026-05-17T15:00:45.126Z,00.json


### E2. Visualizar o Histórico da Tabela

1. Execute o código abaixo para visualizar o histórico da tabela de streaming **orders_bronze_demo2**.

      Observe o seguinte:

      - Na coluna **operation**, as duas últimas atualizações são **STREAMING UPDATE**, não `WRITE` ou `MERGE`. Isso confirma que a tabela está sendo escrita incrementalmente por uma consulta de streaming.

      - Expanda os valores de **operationParameters** das duas últimas atualizações. Note que ambas usam `"outputMode": "Append"` e cada uma possui um **epochId** crescente (`0` para a carga inicial, `1` para a atualização incremental).

      - Encontre a coluna **operationMetrics**. Expanda os valores das duas últimas atualizações. Observe o seguinte:

         - Ela exibe várias métricas da atualização de streaming: **numRemovedFiles, numOutputRows, numOutputBytes** e **numAddedFiles**.

         - Nos valores de `numOutputRows`
            - **174 linhas** foram adicionadas na primeira atualização
            - **25 linhas** na segunda.
            - Se fosse um pipeline em lote fazendo uma releitura completa, a segunda execução mostraria 199 linhas. O fato de mostrar apenas 25 prova que apenas novos dados foram processados.

In [0]:
DESCRIBE HISTORY sdp_1_bronze.orders_bronze_demo2

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-05-17T15:08:10.000Z,73868070869994,labuser15140516_1778971530@vocareum.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 6d33ea8e-e9ba-40c0-8e4a-ee45130e462c, epochId -> 1, statsOnLoad -> true)",null,null,null,0517-145709-x1xu0o0e-v2n,2,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 25, numOutputBytes -> 2394, numAddedFiles -> 1)",null,Databricks-Runtime/dlt:17.3.10-delta-pipelines-aarch64-photon-dlt-release-dp-20260505-rc0-commit-60862fd-image-4a467a3
2,2026-05-17T15:00:47.000Z,73868070869994,labuser15140516_1778971530@vocareum.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 6d33ea8e-e9ba-40c0-8e4a-ee45130e462c, epochId -> 0, statsOnLoad -> true)",null,null,null,0517-145709-x1xu0o0e-v2n,1,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 174, numOutputBytes -> 4352, numAddedFiles -> 1)",null,Databricks-Runtime/dlt:17.3.10-delta-pipelines-aarch64-photon-dlt-release-dp-20260505-rc0-commit-60862fd-image-4a467a3
1,2026-05-17T15:00:39.001Z,73868070869994,labuser15140516_1778971530@vocareum.com,DLT SETUP,"Map(pipelineId -> 1a9d9270-997c-411c-91a8-d8d21c82d7be, updateId -> 3a1dcd1d-2dcd-4e08-9d00-37aa434927e7)",null,null,null,0517-145709-x1xu0o0e-v2n,0,WriteSerializable,false,Map(),null,Databricks-Runtime/dlt:17.3.10-delta-pipelines-aarch64-photon-dlt-release-dp-20260505-rc0-commit-60862fd-image-4a467a3
0,2026-05-17T15:00:39.000Z,73868070869994,labuser15140516_1778971530@vocareum.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""spark.sql.internal.pipelines.parentTableId"":""64b96aab-8353-4be7-847e-b46169e32884"",""delta.enableDeletionVectors"":""true"",""spark.sql.internal.unityCatalog.internalEntity.isListable"":""false"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-2655ee1e-db27-41dc-9f23-e929cb005191"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-f92ac02f-2c64-4634-8a7d-c1616e590b93"",""spark.sql.internal.unityCatalog.internalEntity.inheritsPolicy"":""false"",""delta.writePartitionColumnsToParquet"":""true""}, statsOnLoad -> false)",null,null,null,0517-145709-x1xu0o0e-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/dlt:17.3.10-delta-pipelines-aarch64-photon-dlt-release-dp-20260505-rc0-commit-60862fd-image-4a467a3


## F. Visualizando Pipelines Spark Declarativos com a UI de Pipelines

Após explorar e criar seu pipeline usando o arquivo **orders_pipeline.sql** nos passos acima, você pode visualizar os pipelines criados em seu workspace através da UI **Jobs and Pipelines**.

1. Complete os seguintes passos para visualizar o pipeline que você criou:

   a. No painel de navegação principal à esquerda (talvez seja necessário expandi-lo selecionando o ícone ![Expand Navigation Pane](./Includes/images/developing-a-simple-pipeline/expand_main_navigation.png) no canto superior esquerdo do workspace), clique com o botão direito em **Jobs & Pipelines** e selecione **Abrir link em uma nova aba**.

   b. Isso deve levar você aos pipelines que foram criados. Você deve ver seu pipeline **2 - Developing a Simple Pipeline Project - seu nome**.

   c. Selecione seu pipeline **2 - Developing a Simple Pipeline Project - seu nome**.

   d. Sob o nome do pipeline, selecione o menu suspenso com o timestamp. Aqui você pode visualizar o **Gráfico do Pipeline** e outras métricas de cada execução do pipeline.

   e. Feche a aba da UI de pipelines que você abriu.

   ![Jobs & Pipelines](./Includes/images/developing-a-simple-pipeline/demo_2_view_in_jobs_pipelines.png)

## Additional Resources

- **Lakeflow Spark Declarative Pipelines documentation**:
[AWS](https://docs.databricks.com/aws/en/dlt/) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/) |
[GCP](https://docs.databricks.com/gcp/en/dlt/)


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>